In [65]:
import geopandas as gpd
import joblib
import numpy as np
import odc.geo  # noqa: F401
import pandas as pd
from shapely import geometry
from sklearn.ensemble import RandomForestClassifier

from utils import load_data, safe_divide

In [66]:
%reload_ext autoreload
%autoreload 2

## Find and load S2 data

Load data and set up your array to use for prediction

In [67]:
# Configure some things up front
chunks = dict(x=256, y=256)
datetime = "2022"

bbox_fiji = [177.14, -18.41, 178.80, -17.2]
bbox = [178.14, -17.81, 178.50, -17.4]
bbox_geometry = geometry.box(*bbox)

gdf = gpd.GeoDataFrame({'geometry': [bbox_geometry]}, crs='EPSG:4326')
gdf.explore()

In [68]:
merged = load_data(bbox, chunks=dict(x=256, y=256), resolution=10)

In [69]:
merged

<xarray.Dataset> Size: 3GB
Dimensions:        (y: 4760, x: 4008)
Coordinates:
  * y              (y) float64 38kB -1.955e+06 -1.955e+06 ... -2.002e+06
  * x              (x) float64 32kB 3.133e+06 3.133e+06 ... 3.173e+06 3.173e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2022-01-01
Data variables: (12/28)
    blue           (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    green          (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    red            (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    rededge1       (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    rededge2       (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    rededge3       (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    ...             ...
    slope          (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    aspect         (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    elevation_std  (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    mean_vv        (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    mean_vh        (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    mean_vv_vh     (y, x) float32 76MB dask.array<chunksize=(256, 256), meta=np.ndarray>

## Train and predict

When you change your training data, you can re-train and predict here.

In [70]:
training_file = "training_data/mred-tdata.geojson"

tdata = gpd.read_file(training_file, bbox=bbox_geometry)
tdata.explore()

In [71]:
len(tdata)

197

In [72]:
# Get values for each of the image bands at each of the points.
pts_proj = tdata.to_crs(merged.odc.crs)

# a DataArray with x & y coords
pts_da = pts_proj.assign(x=pts_proj.geometry.x, y=pts_proj.geometry.y).to_xarray()

# a dataframe or series (for a single point)
pt_values_i = (
    merged.sel(pts_da[["x", "y"]], method="nearest").squeeze().compute().to_pandas()
)

if isinstance(pt_values_i, pd.Series):
    pt_values_i = pt_values_i.to_frame().transpose()
    pt_values_i.index = tdata.index

In [73]:
len(pt_values_i)

197

In [74]:
# Check columns of pt_values_i to see what's actually in there
print("pt_values_i columns:", pt_values_i.columns)

# If lulc_code is in tdata, check tdata length and nulls
print("tdata length:", len(tdata))
print("tdata nulls:", tdata['lulc_code'].isnull().sum())

# When merging, ensure you use a left join or check why rows are dropping:
# merged_df = pd.merge(tdata, pt_values_i, left_index=True, right_index=True, how='inner')

pt_values_i columns: Index(['blue', 'green', 'red', 'rededge1', 'rededge2', 'rededge3', 'nir',
       'swir16', 'swir22', 'emad', 'bcmad', 'smad', 'ndvi', 'mndwi', 'evi',
       'savi', 'bsi', 'ndmi', 'ndbi', 'ndwi', 'ndre', 'elevation', 'slope',
       'aspect', 'elevation_std', 'mean_vv', 'mean_vh', 'mean_vv_vh', 'y', 'x',
       'spatial_ref', 'time'],
      dtype='object')
tdata length: 197
tdata nulls: 0


In [75]:
# # Check how many NaNs exist per column in your extracted feature dataframe
# print(pt_values_i.isnull().sum())

# # Check how many rows remain if you dropna only on the target column vs all columns
# print("Valid targets:", pt_values_i['lulc_code'].dropna().shape[0])
# print("Valid complete rows:", dpt_values_i.dropna().shape[0])

In [76]:
concat_df = pd.concat([tdata, pt_values_i], axis=1)

In [77]:
# 1. See which specific columns have the most NaNs
print(concat_df.isnull().sum())

# 2. Clean properly: only drop rows where the target (lulc_code) is missing, 
# or fill/drop based strictly on key features instead of a blind table-wide dropna()
training_array = concat_df.dropna(subset=['lulc_code'])

# If certain raster features have NaNs, you can fill them or drop specifically for modeling:
# feature_cols = ['blue', 'green', 'red', 'ndvi', ...] # your feature list
# training_array = concat_df.dropna(subset=['lulc_code'] + feature_cols)

fid              120
index            120
id               120
x                120
y                120
lulc_code        120
lulc_class       120
layer            126
path             126
geometry         120
blue             120
green            120
red              120
rededge1         120
rededge2         120
rededge3         120
nir              120
swir16           120
swir22           120
emad             120
bcmad            120
smad             120
ndvi             120
mndwi            120
evi              120
savi             120
bsi              120
ndmi             120
ndbi             120
ndwi             120
ndre             120
elevation        120
slope            120
aspect           120
elevation_std    120
mean_vv          120
mean_vh          120
mean_vv_vh       120
y                120
x                120
spatial_ref      120
time             120
dtype: int64


In [78]:
# 1. Drop duplicate columns if pandas created suffixes or repeated coordinates (like 'y.1', 'x.1', etc.)
concat_df = concat_df.loc[:, ~concat_df.columns.duplicated()]

# 2. Verify columns and check where missing values are hiding
print("Cleaned columns:", concat_df.columns.tolist())
print("Null counts per column:\n", concat_df.isnull().sum())

# 3. Safely create your training array by dropping rows only where the target (lulc_code) is missing
training_array = concat_df.dropna(subset=['lulc_code'])

print("Final training array shape:", training_array.shape)

Cleaned columns: ['fid', 'index', 'id', 'x', 'y', 'lulc_code', 'lulc_class', 'layer', 'path', 'geometry', 'blue', 'green', 'red', 'rededge1', 'rededge2', 'rededge3', 'nir', 'swir16', 'swir22', 'emad', 'bcmad', 'smad', 'ndvi', 'mndwi', 'evi', 'savi', 'bsi', 'ndmi', 'ndbi', 'ndwi', 'ndre', 'elevation', 'slope', 'aspect', 'elevation_std', 'mean_vv', 'mean_vh', 'mean_vv_vh', 'spatial_ref', 'time']
Null counts per column:
 fid              120
index            120
id               120
x                120
y                120
lulc_code        120
lulc_class       120
layer            126
path             126
geometry         120
blue             120
green            120
red              120
rededge1         120
rededge2         120
rededge3         120
nir              120
swir16           120
swir22           120
emad             120
bcmad            120
smad             120
ndvi             120
mndwi            120
evi              120
savi             120
bsi              120
ndmi       

In [79]:
training_array = pd.concat([tdata, pt_values_i], axis=1).to_crs(4326)
training_array = training_array.drop(
    columns=[
        "y",
        "x",
        "spatial_ref",
        "time",
        "fid",
        "index",
        "lulc_class",
        "path",
        "geometry",
        "layer",
        "id",
    ]
)
# Drop rows where there are any NaNs
training_array = training_array.dropna()

training_array.head()

,lulc_code,blue,green,red,rededge1,rededge2,rededge3,nir,swir16,swir22,...,ndbi,ndwi,ndre,elevation,slope,aspect,elevation_std,mean_vv,mean_vh,mean_vv_vh
47,1.0,811.0,934.0,936.0,1194.0,1518.0,1656.0,1236.0,1195.0,873.0,...,0.119469,30.061751,0.017284,5.000000,11.388381,285.587036,0.000000,0.052183,0.017979,2.902464
49,1.0,413.0,643.0,323.0,1019.0,3803.0,5025.0,5047.0,1926.0,839.0,...,0.577354,10.743761,0.664029,180.775040,28.120783,146.664017,7.844488,0.236095,0.069069,3.418253
50,1.0,647.0,923.0,664.0,1286.0,2815.0,3302.0,3492.0,1822.0,1058.0,...,0.372836,14.262061,0.461699,15.229146,5.488382,231.532440,1.335231,0.229282,0.044159,5.192220
52,1.0,837.0,874.0,831.0,1102.0,1722.0,1949.0,1155.0,1298.0,895.0,...,0.219547,32.161163,0.023482,14.684795,10.203218,261.407593,0.922163,0.362875,0.038495,9.426587
53,1.0,423.0,725.0,358.0,1134.0,3868.0,4925.0,5089.0,2155.0,986.0,...,0.546581,10.521500,0.635546,215.913071,11.223964,292.107697,0.812147,0.439720,0.083388,5.273170


In [80]:
# # 1. Check feature names
# print("Model expected features:", getattr(model, "feature_names_in_", "None saved"))

# # 2. Check for constant or null features in training data
# for i, col in enumerate(training_array.columns if hasattr(training_array, "columns") else range(training_array.shape[1])):
#     vals = training_array[:, i] if not hasattr(training_array, "columns") else training_array[col]
#     print(f"Feature {col}: min={vals.min()}, max={vals.max()}, nans={np.isnan(vals).sum()}")

In [81]:
training_array.to_csv("training_array.csv")

In [82]:
classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42,
)

training_data = np.array(training_array)[:, 1:]
classes = np.array(training_array)[:, 0]

model = classifier.fit(training_data, classes)


In [83]:
joblib.dump(model, "test_model.dump")

['test_model.dump']

In [84]:
# Print feature importances against column headings
fields_importances = sorted(
    zip(training_array.columns[1:], classifier.feature_importances_),
    key=lambda x: x[1],
    reverse=True,
)

for i in fields_importances:
    # Format as a table to 2 decinal places
    print(f"{i[0]:<11}| {i[1]:.2f}")


bsi        | 0.17
swir22     | 0.07
mean_vh    | 0.07
mean_vv_vh | 0.06
ndre       | 0.05
ndvi       | 0.05
mean_vv    | 0.05
slope      | 0.04
evi        | 0.04
nir        | 0.04
elevation  | 0.04
swir16     | 0.03
savi       | 0.03
aspect     | 0.03
ndwi       | 0.03
rededge1   | 0.03
green      | 0.02
red        | 0.02
bcmad      | 0.02
blue       | 0.02
mndwi      | 0.02
smad       | 0.02
rededge2   | 0.01
elevation_std| 0.01
rededge3   | 0.01
ndmi       | 0.01
emad       | 0.01
ndbi       | 0.00
